# 🤖 Step 3: Interactive Model Training
Training a Random Forest classifier with MLflow experiment tracking and Plotly evaluation metrics.

In [1]:
import pandas as pd
import numpy as np
import plotly.express as px
import mlflow
import mlflow.sklearn
import joblib
import os
from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, f1_score, roc_auc_score, roc_curve, confusion_matrix

# Load features
X = pd.read_csv('../data/processed/features.csv')
y = pd.read_csv('../data/processed/target.csv').values.ravel()

# Split data
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

## 1. Train with MLflow

In [2]:
mlflow.set_experiment('Credit Risk Prediction')
with mlflow.start_run():
    model = RandomForestClassifier(n_estimators=100, max_depth=10, random_state=42)
    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)
    y_prob = model.predict_proba(X_test)[:, 1]
    
    # Log metrics
    metrics = {'accuracy': accuracy_score(y_test, y_pred), 'roc_auc': roc_auc_score(y_test, y_prob)}
    mlflow.log_metrics(metrics)
    mlflow.sklearn.log_model(model, 'model')
    print('Logged to MLflow:', metrics)

2026/03/24 21:50:21 WARNING mlflow.models.model: `artifact_path` is deprecated. Please use `name` instead.
2026/03/24 21:50:21 WARNING mlflow.sklearn: Saving scikit-learn models in the pickle or cloudpickle format requires exercising caution because these formats rely on Python's object serialization mechanism, which can execute arbitrary code during deserialization. The recommended safe alternative is the 'skops' format. For more information, see: https://scikit-learn.org/stable/model_persistence.html


Logged to MLflow: {'accuracy': 0.928030303030303, 'roc_auc': 0.9214410428065752}


## 2. Evaluation Visuals

### 📊 Interpreting Model Evaluation

#### 1. The ROC Curve (Receiver Operating Characteristic)
This curve shows the trade-off between the **True Positive Rate** (finding the defaults) and the **False Positive Rate** (mistakenly labeling a good user as a default).
*   **AUC (Area Under Curve)**: A score from 0 to 1. 
    *   **0.5**: Random guessing.
    *   **0.8+**: Good performance.
    *   **0.9+**: Excellent performance.

#### 2. The Confusion Matrix
This table breaks down exactly where the model succeeded and failed:
*   **Top-Left (TN)**: Correctly predicted "No Default".
*   **Bottom-Right (TP)**: Correctly predicted "Default".
*   **Top-Right (FP)**: **Type I Error** - Predicted "Default" but the user was actually "Good". (Unfair denial).
*   **Bottom-Left (FN)**: **Type II Error** - Predicted "Good" but the user actually "Defaulted". (**Total loss for the bank**).


In [3]:
# 1. ROC Curve with Plotly
fpr, tpr, _ = roc_curve(y_test, y_prob)
fig_roc = px.area(x=fpr, y=tpr, title=f'ROC Curve (AUC={roc_auc_score(y_test, y_prob):.4f})', 
                  labels={'x': 'False Positive Rate (FPR)', 'y': 'True Positive Rate (TPR)'},
                  width=600, height=450)
fig_roc.add_shape(type='line', line=dict(dash='dash'), x0=0, x1=1, y0=0, y1=1)
fig_roc.show()

# 2. Confusion Matrix with explicit labels
cm = confusion_matrix(y_test, y_pred)
fig_cm = px.imshow(cm, text_auto=True, title='Confusion Matrix Heatmap',
                   labels=dict(x='Predicted Status', y='Actual Status'),
                   x=['Non-Default', 'Default'],
                   y=['Non-Default', 'Default'],
                   color_continuous_scale='Blues')
fig_cm.show()

In [ ]:
os.makedirs('../models', exist_ok=True)
joblib.dump(model, '../models/model.joblib')
print('Model saved!')